In [ ]:
import os
import pickle
import torch
import pandas as pd

def compare_pgd_results(results_df):
    """
    Compare PGD with gradient vs without gradient results, grouping by all levels
    of the DataFrame's index automatically.
    
    Parameters:
    results_df (pd.DataFrame): DataFrame with multi-level columns and multi-index
    
    Returns:
    dict: Nested dictionary containing comparison results
    """
    results = {
        'whole_dataset': {
            'L2': {'with_grad > no_grad': 0, 'with_grad < no_grad': 0, 'with_grad == no_grad': 0},
            'Linf': {'with_grad > no_grad': 0, 'with_grad < no_grad': 0, 'with_grad == no_grad': 0}
        },
        'by_index_level': {}
    }

    # Access the multi-level columns properly
    pgd_no_grad_L2 = ('perturbed_rmse', 'pgdnograd_L2')
    pgd_no_grad_Linf = ('perturbed_rmse', 'pgdnograd_Linf')
    pgd_with_grad_L2 = ('perturbed_rmse', 'pgdwithgrad_L2')
    pgd_with_grad_Linf = ('perturbed_rmse', 'pgdwithgrad_Linf')

    # Whole dataset comparison
    results['whole_dataset']['L2']['with_grad > no_grad'] = (results_df[pgd_with_grad_L2] > results_df[pgd_no_grad_L2]).sum()
    results['whole_dataset']['L2']['with_grad < no_grad'] = (results_df[pgd_with_grad_L2] < results_df[pgd_no_grad_L2]).sum()
    results['whole_dataset']['L2']['with_grad == no_grad'] = (results_df[pgd_with_grad_L2] == results_df[pgd_no_grad_L2]).sum()

    results['whole_dataset']['Linf']['with_grad > no_grad'] = (results_df[pgd_with_grad_Linf] > results_df[pgd_no_grad_Linf]).sum()
    results['whole_dataset']['Linf']['with_grad < no_grad'] = (results_df[pgd_with_grad_Linf] < results_df[pgd_no_grad_Linf]).sum()
    results['whole_dataset']['Linf']['with_grad == no_grad'] = (results_df[pgd_with_grad_Linf] == results_df[pgd_no_grad_Linf]).sum()

    # Group by each index level
    for level in range(results_df.index.nlevels):
        level_name = results_df.index.names[level] or f"level_{level}"
        results['by_index_level'][level_name] = {}
        
        for group_val, group_df in results_df.groupby(level=level):
            group_key = f"{level_name}={group_val}"
            results['by_index_level'][level_name][group_key] = {
                'L2': {
                    'with_grad > no_grad': (group_df[pgd_with_grad_L2] > group_df[pgd_no_grad_L2]).sum(),
                    'with_grad < no_grad': (group_df[pgd_with_grad_L2] < group_df[pgd_no_grad_L2]).sum(),
                    'with_grad == no_grad': (group_df[pgd_with_grad_L2] == group_df[pgd_no_grad_L2]).sum()
                },
                'Linf': {
                    'with_grad > no_grad': (group_df[pgd_with_grad_Linf] > group_df[pgd_no_grad_Linf]).sum(),
                    'with_grad < no_grad': (group_df[pgd_with_grad_Linf] < group_df[pgd_no_grad_Linf]).sum(),
                    'with_grad == no_grad': (group_df[pgd_with_grad_Linf] == group_df[pgd_no_grad_Linf]).sum()
                }
            }
    
    return results


def print_pgd_results(results):
    """
    Print the comparison results in the desired format.
    """
    print("\nPGD Results Comparison:")
    print("="*60)
    
    # Print overall comparison first
    print("\nOverall Comparison:")
    print("-"*40)
    print("L2 Norm:")
    print(f"  With grad > No grad: {results['whole_dataset']['L2']['with_grad > no_grad']} cases")
    print(f"  With grad < No grad: {results['whole_dataset']['L2']['with_grad < no_grad']} cases")
    print(f"  With grad == No grad: {results['whole_dataset']['L2']['with_grad == no_grad']} cases")
    
    print("\nLinf Norm:")
    print(f"  With grad > No grad: {results['whole_dataset']['Linf']['with_grad > no_grad']} cases")
    print(f"  With grad < No grad: {results['whole_dataset']['Linf']['with_grad < no_grad']} cases")
    print(f"  With grad == No grad: {results['whole_dataset']['Linf']['with_grad == no_grad']} cases")
    
    # Then print group-wise comparison by index level
    print("\n\nIndex Level Comparison:")
    print("-"*40)
    
    for level_name, level_data in results['by_index_level'].items():
        print(f"\nIndex Level: {level_name}")
        print("-"*30)
        
        # Sort groups naturally
        sorted_groups = sorted(level_data.items(),
                             key=lambda x: float(x[0].split('=')[1]) if x[0].split('=')[1].replace('.','',1).isdigit() else x[0])
        
        for group_key, group_data in sorted_groups:
            print(f"\nGroup {group_key}:")
            print("  L2 Norm:")
            print(f"    With grad > No grad: {group_data['L2']['with_grad > no_grad']} cases")
            print(f"    With grad < No grad: {group_data['L2']['with_grad < no_grad']} cases")
            print(f"    With grad == No grad: {group_data['L2']['with_grad == no_grad']} cases")
            
            print("  Linf Norm:")
            print(f"    With grad > No grad: {group_data['Linf']['with_grad > no_grad']} cases")
            print(f"    With grad < No grad: {group_data['Linf']['with_grad < no_grad']} cases")
            print(f"    With grad == No grad: {group_data['Linf']['with_grad == no_grad']} cases")


# Directory containing the pickle files
folder_path = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/perturbation_results/1D/old_5"  # Replace with your folder path
all_rmse_data = []

for filename in os.listdir(folder_path):
    if filename.endswith(".pkl"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
        
        # Counter for records in the current file
        records_in_file = 0
        
        for item in data:
            if isinstance(item, dict):
                original_rmse = item.get("original deviation RMSE", None)
                perturbed_rmse = item.get("perturbed deviation RMSE", None)
                index = item.get("index", None)
                epsilon = item.get("epsilon", None)
                num = item.get("num_steps", item.get("num_samples", None))
                
                if all(v is not None for v in [original_rmse, perturbed_rmse, index]):
                    all_rmse_data.append({
                        "perturb_type": "_".join(filename.split("_")[:2]),
                        "index": index,
                        "epsilon": epsilon,
                        "num": num,
                        "original_rmse": original_rmse,
                        "perturbed_rmse": perturbed_rmse
                    })
                    records_in_file += 1  # Increment counter for valid records
        
        # Print filename and number of records processed
        print(f"File: {filename} | Records: {records_in_file}")

# Print total number of records after all files are processed
print(f"\nTotal records across all files: {len(all_rmse_data)}")

# Create DataFrame and pivot
df = pd.DataFrame(all_rmse_data)
df_no_rdm = df[df["perturb_type"]!="rdm_none"]
result_df = df_no_rdm.pivot(
    index=['index',"epsilon","num"],
    columns='perturb_type',
    # values=["original_rmse",'perturbed_rmse'],
    values=['perturbed_rmse']
)
results = compare_pgd_results(result_df)
print_pgd_results(results)
result_df

File: pgdwithgrad_Linf_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.03_bc=periodic_nu=0.0005_t=1.0_seed=3_N=10.pkl | Records: 360
File: pgdwithgrad_L2_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.03_bc=periodic_nu=0.0005_t=1.0_seed=3_N=10.pkl | Records: 360
File: pgdnograd_Linf_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.03_bc=periodic_nu=0.0005_t=1.0_seed=3_N=10.pkl | Records: 360
File: pgdnograd_L2_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.03_bc=periodic_nu=0.0005_t=1.0_seed=3_N=10.pkl | Records: 360

Total records across all files: 1440

PGD Results Comparison:

Overall Comparison:
----------------------------------------
L2 Norm:
  With grad > No grad: 132 cases
  With grad < No grad: 228 cases
  With grad == No grad: 0 cases

Linf Norm:
  With grad > No grad: 151 cases
  With grad < No grad: 209 cases
  With grad == No grad: 0 cases


Index Level Comparison:
----------------------------------------

Index Level: index
---

perturbed_rmse                                \
perturb_type        pgdnograd_L2 pgdnograd_Linf pgdwithgrad_L2   
index epsilon num                                                
0     0.01    2     0.0025531114   0.0022955425   0.0025625494   
              5     0.0025526427    0.002278498   0.0025645648   
              10    0.0025531752   0.0022718043   0.0025637436   
              20    0.0025529608   0.0022654878   0.0025639746   
              50    0.0025535612    0.002262439    0.002564646   
...                          ...            ...            ...   
9     0.70    5       0.23351024     0.37011537     0.16452819   
              10      0.20656165      0.3120929     0.13661437   
              20      0.23415312     0.30636227     0.15212412   
              50      0.23431388     0.29096678      0.1508987   
              100     0.24073265     0.28796375     0.14989737   

                                    
perturb_type      pgdwithgrad_Linf  
index epsilon num                   
0     0.01    2       0.0024588099  
              5        0.002450725  
              10      0.0024500755  
              20      0.0024494936  
              50      0.0024484026  
...                            ...  
9     0.70    5         0.38492325  
              10        0.30167413  
              20        0.30118975  
              50        0.32199606  
              100       0.32137072  

[360 rows x 4 columns]

In [4]:
result_df[result_df.index.get_level_values("index")==4]

original_rmse                                \
perturb_type       pgdnograd_L2 pgdnograd_Linf pgdwithgrad_L2   
index epsilon num                                               
4     0.1     2    0.0065543232   0.0065543232   0.0065543232   
              5    0.0065543232   0.0065543232   0.0065543232   
              10   0.0065543232   0.0065543232   0.0065543232   
              20   0.0065543232   0.0065543232   0.0065543232   
              50   0.0065543232   0.0065543232   0.0065543232   
              100  0.0065543232   0.0065543232   0.0065543232   
      0.5     2    0.0065543232   0.0065543232   0.0065543232   
              5    0.0065543232   0.0065543232   0.0065543232   
              10   0.0065543232   0.0065543232   0.0065543232   
              20   0.0065543232   0.0065543232   0.0065543232   
              50   0.0065543232   0.0065543232   0.0065543232   
              100  0.0065543232   0.0065543232   0.0065543232   
      1.0     2    0.0065543232   0.0065543232   0.0065543232   
              5    0.0065543232   0.0065543232   0.0065543232   
              10   0.0065543232   0.0065543232   0.0065543232   
              20   0.0065543232   0.0065543232   0.0065543232   
              50   0.0065543232   0.0065543232   0.0065543232   
              100  0.0065543232   0.0065543232   0.0065543232   

                                   perturbed_rmse                 \
perturb_type      pgdwithgrad_Linf   pgdnograd_L2 pgdnograd_Linf   
index epsilon num                                                  
4     0.1     2       0.0065543232    0.023967337      0.0409061   
              5       0.0065543232    0.023841307     0.04093675   
              10      0.0065543232    0.023941504    0.041006245   
              20      0.0065543232    0.023929907      0.0410742   
              50      0.0065543232    0.023934167    0.041144524   
              100     0.0065543232    0.023933677     0.04118092   
      0.5     2       0.0065543232     0.14687255     0.29799062   
              5       0.0065543232      0.1431611      0.3246283   
              10      0.0065543232     0.14004423     0.31996322   
              20      0.0065543232     0.14292866     0.30220976   
              50      0.0065543232      0.1452323      0.3079875   
              100     0.0065543232     0.14523594     0.31578708   
      1.0     2       0.0065543232       0.280593     0.47016123   
              5       0.0065543232     0.26580733      0.5385607   
              10      0.0065543232     0.33705106     0.64138234   
              20      0.0065543232      0.3599553      0.6327434   
              50      0.0065543232     0.36500672      0.6208079   
              100     0.0065543232     0.37180337     0.56711763   

                                                   
perturb_type      pgdwithgrad_L2 pgdwithgrad_Linf  
index epsilon num                                  
4     0.1     2      0.022595098      0.041433807  
              5      0.022473667      0.041754853  
              10      0.02253347       0.04188501  
              20     0.022527646      0.041935604  
              50     0.022529831      0.041979205  
              100    0.022528833      0.041988257  
      0.5     2       0.13906612       0.26183558  
              5      0.118390545        0.3098248  
              10      0.13144149       0.29202318  
              20       0.1282057       0.29021424  
              50      0.11945937       0.28867352  
              100    0.107338466       0.28771415  
      1.0     2       0.23377804        0.6290581  
              5       0.12633464       0.68741006  
              10      0.19245219        0.6743405  
              20      0.19164206       0.61851263  
              50       0.2029713        0.6035642  
              100     0.20395961       0.61753243

In [45]:
df[df["perturb_type"]=="pgdwithgrad_Linf"]

,perturb_type,index,epsilon,num,original_rmse,perturbed_rmse
0,pgdwithgrad_Linf,0,0.1,2,0.0026132117,0.0026132117
1,pgdwithgrad_Linf,0,0.5,2,0.0026132117,0.0026132117
2,pgdwithgrad_Linf,0,1.0,2,0.0026132117,0.0026132117
3,pgdwithgrad_Linf,0,0.1,5,0.0026132117,0.0026132117
4,pgdwithgrad_Linf,0,0.5,5,0.0026132117,0.0026132117
...,...,...,...,...,...,...
355,pgdwithgrad_Linf,9,0.5,50,0.004900973,0.21207905
356,pgdwithgrad_Linf,9,1.0,50,0.004900973,0.52240705
357,pgdwithgrad_Linf,9,0.1,100,0.004900973,0.026626261
358,pgdwithgrad_Linf,9,0.5,100,0.004900973,0.21096104


In [46]:
df[df["perturb_type"]=="pgdwithgrad_L2"]

,perturb_type,index,epsilon,num,original_rmse,perturbed_rmse
360,pgdwithgrad_L2,0,0.1,2,0.0026132117,0.0026132117
361,pgdwithgrad_L2,0,0.5,2,0.0026132117,0.0026132117
362,pgdwithgrad_L2,0,1.0,2,0.0026132117,0.0026132117
363,pgdwithgrad_L2,0,0.1,5,0.0026132117,0.0026132117
364,pgdwithgrad_L2,0,0.5,5,0.0026132117,0.0026132117
...,...,...,...,...,...,...
535,pgdwithgrad_L2,9,0.5,50,0.004900973,0.004900973
536,pgdwithgrad_L2,9,1.0,50,0.004900973,0.004900973
537,pgdwithgrad_L2,9,0.1,100,0.004900973,0.004900973
538,pgdwithgrad_L2,9,0.5,100,0.004900973,0.004900973


In [40]:
df_no_rdm.dtypes

perturb_type       object
index               int64
epsilon           float64
num                 int64
original_rmse      object
perturbed_rmse     object
dtype: object

In [42]:
df_no_rdm[["perturb_type",'index',"epsilon","num"]].duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
1075    False
1076    False
1077    False
1078    False
1079    False
Length: 1080, dtype: bool

In [43]:
df_no_rdm[df_no_rdm[["perturb_type",'index',"epsilon","num"]].duplicated()]

,perturb_type,index,epsilon,num,original_rmse,perturbed_rmse
180,pgdwithgrad_Linf,0,0.1,2,0.0026132117,0.011931404
181,pgdwithgrad_Linf,0,0.5,2,0.0026132117,0.24888656
182,pgdwithgrad_Linf,0,1.0,2,0.0026132117,0.64616096
183,pgdwithgrad_Linf,0,0.1,5,0.0026132117,0.011793734
184,pgdwithgrad_Linf,0,0.5,5,0.0026132117,0.24774072
...,...,...,...,...,...,...
895,pgdnograd_Linf,9,0.5,50,0.004900973,0.21145917
896,pgdnograd_Linf,9,1.0,50,0.004900973,0.5150424
897,pgdnograd_Linf,9,0.1,100,0.004900973,0.035997555
898,pgdnograd_Linf,9,0.5,100,0.004900973,0.20911245


In [36]:
df_no_rdm[df_no_rdm.duplicated()]

TypeError: unhashable type: 'jaxlib.xla_extension.ArrayImpl'

In [37]:
df_no_rdm.duplicated()

TypeError: unhashable type: 'jaxlib.xla_extension.ArrayImpl'

In [38]:
df_no_rdm.dtypes()

TypeError: 'Series' object is not callable

In [32]:
df

,perturb_type,index,epsilon,num,original_rmse,perturbed_rmse
0,pgdwithgrad_Linf,0,0.1,2,0.0026132117,0.0026132117
1,pgdwithgrad_Linf,0,0.5,2,0.0026132117,0.0026132117
2,pgdwithgrad_Linf,0,1.0,2,0.0026132117,0.0026132117
3,pgdwithgrad_Linf,0,0.1,5,0.0026132117,0.0026132117
4,pgdwithgrad_Linf,0,0.5,5,0.0026132117,0.0026132117
...,...,...,...,...,...,...
1255,rdm_none,9,0.5,10000,0.004900973,0.045408096
1256,rdm_none,9,1.0,10000,0.004900973,0.12421721
1257,rdm_none,9,0.1,10000,0.004900973,0.006084247
1258,rdm_none,9,0.5,10000,0.004900973,0.046110403


In [31]:
df[["perturb_type",'index',"epsilon","num"]][df["perturb_type"]!="rdm_none"].duplicated().sum()

np.int64(360)

In [29]:
df[["perturb_type",'index',"epsilon","num"]]

,perturb_type,index,epsilon,num
0,pgdwithgrad_Linf,0,0.1,2
1,pgdwithgrad_Linf,0,0.5,2
2,pgdwithgrad_Linf,0,1.0,2
3,pgdwithgrad_Linf,0,0.1,5
4,pgdwithgrad_Linf,0,0.5,5
...,...,...,...,...
1255,rdm_none,9,0.5,10000
1256,rdm_none,9,1.0,10000
1257,rdm_none,9,0.1,10000
1258,rdm_none,9,0.5,10000


In [23]:
data[3]

{'index': 0,
 'epsilon': 0.1,
 'num_samples': 10000,
 'max_rmse': array(0.22950977, dtype=float32),
 'dict_len': 2664,
 'a': array([0.08122416, 0.07658845, 0.07200625, ..., 0.09533535, 0.09063102,
        0.08591133], shape=(1024,)),
 'delta': array([-0.08915817, -0.02602785, -0.02818381, ...,  0.0533872 ,
         0.05890659, -0.07899453], shape=(1024,), dtype=float32),
 'a+delta': array([-0.007934  ,  0.0505606 ,  0.04382244, ...,  0.14872256,
         0.14953762,  0.00691679], shape=(1024,), dtype=float32),
 'g(a)': Array([0.33459055, 0.33611998, 0.33764586, ..., 0.3300184 , 0.3315466 ,
        0.3330711 ], dtype=float32),
 'G(a)': array([0.33429855, 0.3358634 , 0.33742672, ..., 0.3295824 , 0.33108503,
        0.33263713], shape=(1024,), dtype=float32),
 'g(a+delta)': Array([0.33400345, 0.33546677, 0.33693177, ..., 0.32962027, 0.33107433,
        0.3325298 ], dtype=float32),
 'G(a+delta)': array([0.3337713 , 0.33619198, 0.33771223, ..., 0.33100313, 0.33263898,
        0.33223382], s

In [13]:
! pip install pandas

/blue/shiboli.fsu/yifeisun.umich/.conda/envs/adv_robust/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [4]:
os.listdir(folder_path)

['1D']

In [ ]:
def convert_to_cpu_serializable(data):
    if isinstance(data, dict):
        return {k: convert_to_cpu_serializable(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [convert_to_cpu_serializable(v) for v in data]
    elif isinstance(data, torch.Tensor):
        return data.cpu().detach().numpy()
    else:
        return data

folder_path = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/perturbation_results/1D"    
for filename in os.listdir(folder_path):
    if filename.endswith(".pkl"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'rb') as f:
            # data = pickle.load(f)
            data = torch.load(f, map_location=torch.device('cpu'), weights_only=False)   
            

In [2]:
import pickle
import os
from pathlib import Path

def combine_pickle_lists(folder1, folder2, output_folder):
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Get all pickle files in both folders
    files1 = {f.name: f for f in Path(folder1).glob('*.pkl')}
    files2 = {f.name: f for f in Path(folder2).glob('*.pkl')}
    
    # Find common filenames
    common_files = set(files1.keys()) & set(files2.keys())
    
    for filename in common_files:
        # Load lists from both folders
        with open(files1[filename], 'rb') as f:
            list1 = pickle.load(f)
        with open(files2[filename], 'rb') as f:
            list2 = pickle.load(f)
        
        # Combine lists (assuming they are lists)
        combined_list = list1 + list2
        
        # Save combined list to output folder
        output_path = Path(output_folder) / filename
        with open(output_path, 'wb') as f:
            pickle.dump(combined_list, f)
        
        print(f"Combined and saved {filename}")

# Example usage
folder1 = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/perturbation_results/1D/"
folder2 = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/perturbation_results/1D/old_4"
output_folder = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/perturbation_results/1D/old_5"

combine_pickle_lists(folder1, folder2, output_folder)

Combined and saved pgdnograd_Linf_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.03_bc=periodic_nu=0.0005_t=1.0_seed=3_N=10.pkl
Combined and saved pgdwithgrad_L2_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.03_bc=periodic_nu=0.0005_t=1.0_seed=3_N=10.pkl
Combined and saved pgdwithgrad_Linf_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.03_bc=periodic_nu=0.0005_t=1.0_seed=3_N=10.pkl
Combined and saved pgdnograd_L2_dim=1d_solver=exponax_kernel=gaussian_correlation_length=0.03_bc=periodic_nu=0.0005_t=1.0_seed=3_N=10.pkl
